# XFeat — 사분면 관계 가설과 역방향 검증 (2026-10-06)

**새 아이디어와 원본 float cosine 기준선만 구현했습니다. 런타임 → 모두 실행.** 기존 노트북은 변경하지 않습니다.

1. A의 왼쪽 위 사분면 전체를 B 전체와 cosine 비교. 빈 사분면이면 가장 많은 KP가 있는 영역 사용.
2. 높은 유사도와 1·2위 점수 차이를 만족하는 임시 대응으로 **평행이동 / 회전+축척+이동(similarity)** 두 가설 생성.
3. 역변환 예측 차이가 큰 곳부터 공간적으로 분산된 B 검증점 최대 16개 선택. B→A 전체 검색으로 독립적인 외부 영역 증거 확인. 기존 점수 재사용.
4. 가설 통과 시 변환된 A 사분면의 범위와 겹치는 B 사분면을 검색. 둘 다 통과하면 합집합. 모두 실패하면 전체 검색으로 복귀.
5. 초기 검색·검증·추가 검색의 **모든 개별 KP 점수**를 합쳐 최종 MNN. 후보 집합 안에서의 MNN이며 전체 검색과 동일하다는 보장은 없습니다.

**사분면 번호:** 1 왼쪽 위, 2 오른쪽 위, 3 오른쪽 아래, 4 왼쪽 아래. 이동 시 경계를 넘을 수 있으므로 항상 1:1 사분면 대응으로 강제하지 않습니다.

**출력:** geometric precision@3px, 정답 수, baseline 정답 매칭 유지율, 후보 유지율, 실제 비교 pair 비율, fallback 비율, 총시간·단계별 시간·p10/p90, 행렬곱 호출 수, 점수 캐시 크기, CPU/라이브러리/thread 설정.

**측정 범위:** 정규화된 XFeat descriptor 이후부터 최종 MNN까지. 초기 탐색·모델 적합·검증·fallback·할당 포함. XFeat 추출·threshold calibration·정답 검증·파일 저장 제외.

**주의:** 빠른 구현을 위해 N×M float32 점수 캐시를 유지합니다. 연산량 감소 실험이며 메모리 절감은 주장하지 않습니다. 가설 검증/작은 행렬곱의 비용으로 원본보다 느릴 수도 있습니다.

Calibration은 COCO의 별도 18장, 검증은 나머지 6장. descriptor threshold만 calibration에서 고정하고 geometry 설정은 명시된 실험 기본값입니다. HPatches 정답은 matcher에 전달하지 않습니다. 합성 warp RNG는 독립적으로 고정했으므로 과거 노트북과 절대 시간/매칭 수를 직접 비교하지 말고 이 파일의 baseline과 비교하세요.


## 1. 환경·데이터 설정
CPU_THREADS=2, TOP_K=800. 기존 Drive 데이터 ZIP 재사용. HPatches는 독립 테스트이며 `RUN_HPatches=False`로 생략할 수 있습니다.

In [1]:
%pip -q install opencv-python-headless pandas threadpoolctl tqdm


import time
import io
import zipfile
import subprocess
import shutil
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from google.colab import files, drive

SEED = 7
TOP_K = 800
CALIBRATION_WARPS_PER_IMAGE = 1
VALIDATION_WARPS_PER_IMAGE = 3
GEOMETRIC_THRESHOLD = 3.0
CPU_THREADS = 2
RUN_HPatches = True
TIMING_PAIRS_PER_DATASET = 5  # evenly spaced pairs; compare every method on each pair
WARMUP = 5
REPEATS = 12
TIMING_ROUNDS = 3

torch.torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

xfeat = torch.hub.load('verlab/accelerated_features', 'XFeat', pretrained=True, top_k=TOP_K, trust_repo=True).to(device).eval()
for parameter in xfeat.parameters():
    parameter.requires_grad_(False)
print('Device:', device)
# COCO is used for calibration and an internal synthetic-warp check.
# Compression/Blob models use calibration scenes. HPatches is reserved for external evaluation.
DATA_SOURCE = 'coco'  # change to 'upload' to use your own JPG/PNG files
COCO_SOURCE_COUNT = 24
COCO_URL = 'https://images.cocodataset.org/zips/val2017.zip'
drive.mount('/content/drive')
data_dir = Path('/content/drive/MyDrive/Colab Notebooks/CSE527_datasets')
data_dir.mkdir(parents=True, exist_ok=True)
print('Persistent dataset cache:', data_dir)

def download_archive(url, destination):
    if destination.exists() and zipfile.is_zipfile(destination):
        print('Using persistent Drive cache:', destination)
        return
    # Reuse a download from an earlier run in this same Colab session, if present.
    legacy = Path('/content/xfeat_data') / destination.name
    if legacy.exists() and zipfile.is_zipfile(legacy):
        shutil.copy2(legacy, destination)
        print('Copied previous runtime download into Drive:', destination)
        return
    print('Downloading once into Drive:', url)
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', str(destination), url], check=True)
    if not zipfile.is_zipfile(destination):
        raise RuntimeError(f'Incomplete or invalid ZIP: {destination}. Delete it and rerun this cell.')

if DATA_SOURCE == 'coco':
    coco_zip = data_dir / 'val2017.zip'
    download_archive(COCO_URL, coco_zip)
    with zipfile.ZipFile(coco_zip) as archive:
        names = sorted(n for n in archive.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png')))
        if len(names) < COCO_SOURCE_COUNT:
            raise RuntimeError('COCO archive has fewer images than requested.')
        selected = np.random.default_rng(SEED).choice(len(names), COCO_SOURCE_COUNT, replace=False)
        image_items = [(Path(names[i]).name, archive.read(names[i])) for i in selected]
    print(f'Selected {len(image_items)} COCO images using seed {SEED}; no COCO annotations needed.')
elif DATA_SOURCE == 'upload':
    print('Upload JPG/PNG images from distinct scenes.')
    uploaded = files.upload()
    image_items = list(uploaded.items())
    if not image_items:
        raise ValueError('Upload at least one image.')
else:
    raise ValueError("DATA_SOURCE must be 'coco' or 'upload'")

def decode_and_resize(raw_bytes, maximum_side=640):
    image = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError('An uploaded file is not a readable JPG/PNG image.')
    height, width = image.shape[:2]
    scale = min(1.0, maximum_side / max(height, width))
    new_width = max(64, (int(width * scale) // 32) * 32)
    new_height = max(64, (int(height * scale) // 32) * 32)
    return cv2.resize(image, (new_width, new_height))

# Split by source image, before any synthetic warps or descriptor calibration.
# Upload images of distinct scenes; near-duplicate photos belong to the same split.
image_items = sorted(image_items)
images = [decode_and_resize(raw) for _, raw in image_items]
if len(images) >= 2:
    split_rng = np.random.default_rng(SEED)
    order = split_rng.permutation(len(images))
    validation_count = max(1, int(np.ceil(0.25 * len(images))))
    validation_ids = set(order[:validation_count].tolist())
    calibration_images = [im for i, im in enumerate(images) if i not in validation_ids]
    validation_images = [im for i, im in enumerate(images) if i in validation_ids]
    print('Calibration files:', [name for i, (name, _) in enumerate(image_items) if i not in validation_ids])
    print('Held-out files:', [name for i, (name, _) in enumerate(image_items) if i in validation_ids])
else:
    raise ValueError('At least two distinct source images are required.')
print('Calibration images:', len(calibration_images), 'Validation images:', len(validation_images))

Downloading: "https://github.com/verlab/accelerated_features/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/verlab/accelerated_features/raw/main/weights/xfeat.pt" to /root/.cache/torch/hub/checkpoints/xfeat.pt


100%|██████████| 5.96M/5.96M [00:00<00:00, 245MB/s]


Device: cpu
Mounted at /content/drive
Persistent dataset cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets
Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/val2017.zip
Selected 24 COCO images using seed 7; no COCO annotations needed.
Calibration files: ['000000002261.jpg', '000000029675.jpg', '000000067534.jpg', '000000162858.jpg', '000000165713.jpg', '000000173008.jpg', '000000175364.jpg', '000000194506.jpg', '000000270402.jpg', '000000289343.jpg', '000000360097.jpg', '000000460841.jpg', '000000471869.jpg', '000000479596.jpg', '000000504711.jpg', '000000517832.jpg', '000000528524.jpg', '000000547383.jpg']
Held-out files: ['000000077595.jpg', '000000132544.jpg', '000000334006.jpg', '000000394559.jpg', '000000449432.jpg', '000000474039.jpg']
Calibration images: 18 Validation images: 6


## 2. XFeat 추출과 합성 이미지 쌍
추출 시간은 매칭 속도에서 제외합니다. 학습 모델은 추가하지 않습니다.

In [2]:
def image_to_tensor(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb.copy()).permute(2, 0, 1).float()[None].to(device) / 255.0

@torch.no_grad()
def extract_features(image):
    output = xfeat.detectAndCompute(image_to_tensor(image), top_k=TOP_K)[0]
    return {
        'keypoints': output['keypoints'].detach().cpu().float(),
        'descriptors': F.normalize(output['descriptors'].detach().cpu().float(), dim=-1),
    }

def make_random_homography(height, width, generator):
    center = (width / 2.0, height / 2.0)
    angle = generator.uniform(-20.0, 20.0)
    scale = generator.uniform(0.88, 1.08)
    affine = cv2.getRotationMatrix2D(center, angle, scale)
    affine[0, 2] += generator.uniform(-0.06, 0.06) * width
    affine[1, 2] += generator.uniform(-0.06, 0.06) * height
    homography = np.vstack([affine, [0., 0., 1.]]).astype(np.float32)
    homography[2, 0] = generator.uniform(-7e-5, 7e-5)
    homography[2, 1] = generator.uniform(-7e-5, 7e-5)
    return homography

def warp_image(image, homography, generator):
    height, width = image.shape[:2]
    warped = cv2.warpPerspective(image, homography, (width, height),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    alpha = float(generator.uniform(0.82, 1.12))
    beta = float(generator.uniform(-12., 18.))
    return np.clip(warped.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

def project_points(points, homography):
    p = points.detach().cpu().numpy()
    p_h = np.concatenate([p, np.ones((len(p), 1), np.float32)], axis=1)
    projected = (homography @ p_h.T).T
    return torch.from_numpy((projected[:, :2] / projected[:, 2:3]).astype(np.float32))

def build_pairs(source_images,warps_per_image,generator,label):
    pairs=[]
    for image_index,image in enumerate(source_images):
        base=extract_features(image)
        for warp_index in range(warps_per_image):
            H=make_random_homography(*image.shape[:2],generator)
            warped=extract_features(warp_image(image,H,generator))
            pairs.append(dict(d1=base['descriptors'],d2=warped['descriptors'],
                k1=base['keypoints'],k2=warped['keypoints'],H=H,shape1=image.shape[:2],shape2=image.shape[:2],
                name=f'{label}_{image_index}_{warp_index}'))
    return pairs
start=time.perf_counter()
calibration_pairs=build_pairs(calibration_images,CALIBRATION_WARPS_PER_IMAGE,np.random.default_rng(SEED),'calibration')
validation_pairs=build_pairs(validation_images,VALIDATION_WARPS_PER_IMAGE,np.random.default_rng(SEED+1),'validation')
print('Extraction seconds:',time.perf_counter()-start)
print('Calibration pairs:',len(calibration_pairs),'Synthetic validation pairs:',len(validation_pairs))


Extraction seconds: 16.77192165500003
Calibration pairs: 18 Synthetic validation pairs: 18


## 3. HPatches 실제 촬영 이미지 쌍 (선택)

In [3]:
real_pairs = []
if RUN_HPatches:
    HPATCHES_URL = 'https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip'
    HPATCHES_SEQUENCES_PER_TYPE = 6  # 6 viewpoint + 6 illumination scenes
    HPATCHES_TARGETS = (2, 3)       # two real photos per sequence
    hpatches_zip = data_dir / 'hpatches-sequences-release.zip'
    download_archive(HPATCHES_URL, hpatches_zip)

    def decode_hpatches_with_scale(raw_bytes):
        original = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
        if original is None:
            raise RuntimeError('Failed to decode HPatches PPM image')
        old_h, old_w = original.shape[:2]
        resized = decode_and_resize(raw_bytes)
        new_h, new_w = resized.shape[:2]
        scale = np.diag([new_w / old_w, new_h / old_h, 1.0]).astype(np.float64)
        return resized, scale

    real_pairs = []
    with zipfile.ZipFile(hpatches_zip) as archive:
        entries = set(archive.namelist())
        folders = sorted({name.rsplit('/', 1)[0] for name in entries if name.endswith('/1.ppm')})
        for prefix in ('v_', 'i_'):
            selected_folders = [folder for folder in folders if folder.split('/')[-1].startswith(prefix)]
            if len(selected_folders) < HPATCHES_SEQUENCES_PER_TYPE:
                raise RuntimeError(f'Expected more HPatches {prefix} folders; found {len(selected_folders)}')
            for folder in selected_folders[:HPATCHES_SEQUENCES_PER_TYPE]:
                image_1, scale_1 = decode_hpatches_with_scale(archive.read(folder + '/1.ppm'))
                features_1 = extract_features(image_1)
                for target_id in HPATCHES_TARGETS:
                    image_name = f'{folder}/{target_id}.ppm'
                    h_name = f'{folder}/H_1_{target_id}'
                    if image_name not in entries or h_name not in entries:
                        raise RuntimeError(f'Missing HPatches image or homography: {folder}, {target_id}')
                    image_2, scale_2 = decode_hpatches_with_scale(archive.read(image_name))
                    features_2 = extract_features(image_2)
                    original_h = np.loadtxt(io.BytesIO(archive.read(h_name))).reshape(3, 3)
                    resized_h = scale_2 @ original_h @ np.linalg.inv(scale_1)
                    real_pairs.append({
                        'name': f'{folder.split("/")[-1]}:1->{target_id}',
                        'group': prefix, 'd1': features_1['descriptors'],
                        'd2': features_2['descriptors'],
                        'k1': features_1['keypoints'], 'k2': features_2['keypoints'],
                        'H': resized_h, 'shape1': image_1.shape[:2], 'shape2': image_2.shape[:2],
                    })
    print('HPatches real-photo pairs:', len(real_pairs), '(viewpoint and illumination)')



Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/hpatches-sequences-release.zip
HPatches real-photo pairs: 24 (viewpoint and illumination)


## 4. 사분면 관계 matcher
정답 H를 입력받지 않습니다. 평행이동은 RANSAC displacement, similarity는 두 점으로 시작해 inlier least squares로 보정합니다. 의심스러운 초기 대응/역방향 증거 부족은 전체 검색으로 복귀합니다.

In [4]:
"""Float-only spatial hypothesis matcher. No ground truth enters the matcher."""
import numpy as np
import time
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class RelationConfig:
    anchor_quadrant: int = 0       # clockwise: 0 TL, 1 TR, 2 BR, 3 BL
    cosine_min: float = 0.50
    margin_min: float = 0.02
    min_seed: int = 8
    geom_tol: float = 0.035        # fraction of image max(height,width)
    region_padding: float = 0.04
    min_seed_fraction: float = 0.45
    min_seed_span: float = 0.08
    probes: int = 16
    min_probe_matches: int = 4
    min_probe_fraction: float = 0.60
    ransac_trials: int = 96
    min_scale: float = 0.5
    max_scale: float = 2.0
    seed: int = 7


def quadrant_ids(k,shape):
    h,w=shape
    right=k[:,0]>=w/2;bottom=k[:,1]>=h/2
    return np.where(bottom,np.where(right,2,3),np.where(right,1,0)).astype(np.int64)


def xy_complex(k,shape):
    k=np.asarray(k,dtype=np.float64)/max(shape)
    return k[:,0]+1j*k[:,1]


def confident_rows(scores,cfg):
    if scores.shape[1]<2:return np.zeros(scores.shape[0],bool),np.zeros(scores.shape[0],np.int64)
    j=scores.argmax(1)
    top=scores[np.arange(len(scores)),j]
    second=np.partition(scores,-2,axis=1)[:,-2]
    return (top>=cfg.cosine_min)&((top-second)>=cfg.margin_min),j


def mnn_scores(s):
    n,m=s.shape
    if not n or not m:return np.empty(0,np.int64),np.empty(0,np.int64)
    r=s.argmax(1);c=s.argmax(0);i=np.arange(n)
    good=np.isfinite(s[i,r])&(c[r]==i)
    return i[good],r[good]


def fit_hypotheses(p,q,cfg):
    """Robust translation and 2D similarity (rotation + scale + translation)."""
    if len(p)<cfg.min_seed:return []
    rng=np.random.default_rng(cfg.seed)
    hypotheses=[]
    for kind in ('translation','similarity'):
        best=None
        for trial in range(cfg.ransac_trials):
            if kind=='translation':
                t=q[trial%len(p)]-p[trial%len(p)];alpha=1+0j
            else:
                i,j=rng.choice(len(p),2,replace=False)
                if abs(p[i]-p[j])<cfg.min_seed_span:continue
                alpha=(q[j]-q[i])/(p[j]-p[i])
                if not cfg.min_scale<=abs(alpha)<=cfg.max_scale:continue
                t=q[i]-alpha*p[i]
            e=np.abs(alpha*p+t-q);mask=e<=cfg.geom_tol
            key=(int(mask.sum()),-float(np.median(e[mask])) if mask.any() else -np.inf)
            if best is None or key>best[0]:best=(key,mask,alpha,t)
        if best is None:continue
        mask=best[1]
        if mask.sum()<cfg.min_seed or mask.mean()<cfg.min_seed_fraction:continue
        pp,qq=p[mask],q[mask]
        span=float(np.hypot(np.ptp(pp.real),np.ptp(pp.imag)))
        if span<cfg.min_seed_span:continue
        if kind=='translation':
            delta=qq-pp;alpha=1+0j;t=np.median(delta.real)+1j*np.median(delta.imag)
        else:
            pc=pp-pp.mean();qc=qq-qq.mean();den=(abs(pc)**2).sum()
            if den<1e-12:continue
            alpha=(np.conj(pc)*qc).sum()/den;t=qq.mean()-alpha*pp.mean()
            if not cfg.min_scale<=abs(alpha)<=cfg.max_scale:continue
        e=abs(alpha*p+t-q);mask=e<=cfg.geom_tol
        if mask.sum()<cfg.min_seed or mask.mean()<cfg.min_seed_fraction:continue
        hypotheses.append(dict(kind=kind,alpha=alpha,t=t,seed_inliers=int(mask.sum()),
            seed_fraction=float(mask.mean()),seed_error=float(np.median(e[mask]))))
    return hypotheses


def choose_probes(zb,models,budget):
    """Spread probes; first probe maximizes inverse-hypothesis disagreement."""
    if not len(zb):return np.empty(0,np.int64)
    if len(models)>1:
        predictions=np.stack([(zb-h['t'])/h['alpha'] for h in models])
        disagreement=np.max(abs(predictions-predictions[0]),axis=0)
    else:disagreement=abs(zb-zb.mean())
    chosen=[];available=np.ones(len(zb),bool);spread=np.ones(len(zb))
    scale=max(float(disagreement.max()),1e-12)
    for _ in range(min(budget,len(zb))):
        priority=disagreement/scale if not chosen else spread*(1+disagreement/scale)
        j=int(np.argmax(np.where(available,priority,-np.inf)))
        chosen.append(j);available[j]=False
        spread=np.minimum(spread,abs(zb-zb[j]))
    return np.array(chosen,np.int64)


def quadrant_rects(shape):
    h,w=np.array(shape,dtype=float)/max(shape)
    return [(0,w/2,0,h/2),(w/2,w,0,h/2),(w/2,w,h/2,h),(0,w/2,h/2,h)]


def region_graph(models,shape1,shape2,padding):
    # Union if both hypotheses pass. Bounding boxes are conservative for rotated rectangles.
    graph=np.zeros((4,4),bool)
    for h in models:
        for a,(x0,x1,y0,y1) in enumerate(quadrant_rects(shape1)):
            corners=np.array([x0+1j*y0,x1+1j*y0,x1+1j*y1,x0+1j*y1])
            z=h['alpha']*corners+h['t']
            lo_x,hi_x=z.real.min()-padding,z.real.max()+padding
            lo_y,hi_y=z.imag.min()-padding,z.imag.max()+padding
            for b,(u0,u1,v0,v1) in enumerate(quadrant_rects(shape2)):
                graph[a,b]|=(hi_x>=u0 and lo_x<=u1 and hi_y>=v0 and lo_y<=v1)
    return graph


def relation_match(x,y,k1,k2,shape1,shape2,cfg,trace=False):
    """Inputs are contiguous normalized float32 descriptors; H is intentionally absent.
    Score cache is dense float32, with -inf for unsearched edges. Memory reduction
    is NOT claimed. Disjoint rectangles ensure no pair is scored twice.
    """
    total_start=time.perf_counter_ns();stages={};last=total_start
    def mark(name):
        nonlocal last
        now=time.perf_counter_ns();stages[name]=(now-last)/1e6;last=now
    n,m=len(x),len(y);q1=quadrant_ids(k1,shape1);q2=quadrant_ids(k2,shape2)
    s=np.full((n,m),-np.inf,np.float32)
    count=0;calls=0;max_block=0
    edges=np.zeros((n,m),bool) if trace else None
    def score(rows,cols):
        nonlocal count,calls,max_block
        if not len(rows) or not len(cols):return
        ix=np.ix_(rows,cols)
        if trace:
            assert not edges[ix].any(),'Duplicate pair evaluation'
            edges[ix]=True
        block=x[rows]@y[cols].T
        s[ix]=block
        count+=len(rows)*len(cols);calls+=1;max_block=max(max_block,block.nbytes)
    counts=np.bincount(q1,minlength=4)
    anchor=cfg.anchor_quadrant
    if counts[anchor]==0:anchor=int(counts.argmax())
    ar=np.flatnonzero(q1==anchor);rest=np.flatnonzero(q1!=anchor);all_b=np.arange(m)
    models=[];accepted=[];probe_ids=np.empty(0,np.int64);fallback=''
    mark('layout_cache_ms')
    score(ar,all_b)
    mark('anchor_search_ms')
    if not n or not m:
        fallback='empty_input'
    else:
        good,j=confident_rows(s[ar],cfg)
        # Reciprocal within the anchor block only: provisional seeds, NOT final MNN.
        winners=s[ar].argmax(0)
        good &= winners[j]==np.arange(len(ar))
        za,zb=xy_complex(k1,shape1),xy_complex(k2,shape2)
        models=fit_hypotheses(za[ar[good]],zb[j[good]],cfg)
        if not models:fallback='insufficient_seed_geometry'
        else:probe_ids=choose_probes(zb,models,cfg.probes)
    mark('hypothesis_fit_ms')
    # Probe columns are fully searched B->A; anchor rows already cached.
    score(rest,probe_ids)
    mark('reverse_probe_search_ms')
    if models:
        good,ii=confident_rows(s[:,probe_ids].T,cfg)
        # Require validation evidence outside the original anchor region.
        good &= q1[ii]!=anchor
        for h in models:
            residual=abs(h['alpha']*za[ii]+h['t']-zb[probe_ids])
            inliers=good&(residual<=cfg.geom_tol)
            h['probe_usable']=int(good.sum());h['probe_inliers']=int(inliers.sum())
            h['probe_fraction']=float(inliers.sum()/max(1,good.sum()))
            if inliers.sum()>=cfg.min_probe_matches and h['probe_fraction']>=cfg.min_probe_fraction:accepted.append(h)
        if not accepted:fallback='reverse_validation_failed'
    graph=region_graph(accepted,shape1,shape2,cfg.region_padding) if accepted else np.ones((4,4),bool)
    # An unpredicted A quadrant gets full search instead of an empty candidate set.
    for a in range(4):
        if counts[a] and not graph[a].any():graph[a]=True
    mark('validation_graph_ms')
    unprobed=np.ones(m,bool);unprobed[probe_ids]=False
    for a in range(4):
        if a==anchor:continue
        rows=np.flatnonzero(q1==a)
        cols=np.flatnonzero(unprobed&graph[a,q2])
        score(rows,cols)
    mark('remaining_search_ms')
    ij=mnn_scores(s)
    mark('final_mnn_ms')
    result=dict(matches=ij,compared_pairs=count,total_pairs=n*m,matmul_calls=calls,
        cache_bytes=s.nbytes,max_score_block_bytes=max_block,anchor=anchor,
        anchor_kps=len(ar),probe_kps=len(probe_ids),fallback=bool(fallback),fallback_reason=fallback,
        models=[dict(kind=h['kind'],scale=float(abs(h['alpha'])),angle_deg=float(np.degrees(np.angle(h['alpha']))),
            tx=float(h['t'].real),ty=float(h['t'].imag),seed_inliers=h['seed_inliers'],
            probe_inliers=h.get('probe_inliers',0),probe_usable=h.get('probe_usable',0),
            accepted=any(h is a for a in accepted)) for h in models],
        graph=graph,stages=stages)
    result['total_ms_internal']=(time.perf_counter_ns()-total_start)/1e6
    if trace:result.update(scores=s,edges=edges)
    return result


def full_cosine(x,y):return mnn_scores(x@y.T)


## 5. 구현 정확성 검사
이동·회전·축소·복합변환·빈 입력·모호한 descriptor·fallback·중복 계산 방지 검사. 이 검사는 실제 사진의 속도/정확도 측정이 아닙니다.

In [5]:
def check_relation_implementation():
    rng=np.random.default_rng(91);cfg=RelationConfig()
    for label,alpha,t in [('shift',1+0j,.12+.07j),('rotation',1j,1+0j),
                          ('shrink',.7+0j,.15+.15j),('combined',.8*np.exp(.5j),.25+.05j)]:
        z=rng.uniform(.05,.75,180)+1j*rng.uniform(.05,.75,180)
        w=alpha*z+t;valid=(w.real>0)&(w.real<1)&(w.imag>0)&(w.imag<1);z=z[valid];w=w[valid]
        x=rng.normal(size=(len(z),64)).astype(np.float32);x/=np.linalg.norm(x,axis=1,keepdims=True)
        perm=rng.permutation(len(z));y=x[perm].copy()
        k1=np.c_[z.real,z.imag]*1000;k2=np.c_[w[perm].real,w[perm].imag]*1000
        r=relation_match(x,y,k1,k2,(1000,1000),(1000,1000),cfg,trace=True)
        dense=x@y.T;mask=r['edges']
        np.testing.assert_allclose(r['scores'][mask],dense[mask],atol=2e-6,rtol=1e-5)
        for a,b in zip(r['matches'],mnn_scores(np.where(mask,dense,-np.inf))):np.testing.assert_array_equal(a,b)
        for a,b in zip(r['matches'],full_cosine(x,y)):np.testing.assert_array_equal(a,b)
        assert not r['fallback'],(label,r['fallback_reason'])
        assert r['compared_pairs']==mask.sum()<=len(x)*len(y)
        if label=='rotation':assert any(h['kind']=='similarity' and h['accepted'] for h in r['models'])
    # Ambiguous/repeated descriptors must trigger full-search fallback; ties use smallest index.
    x=np.ones((17,64),np.float32)/8;y=np.ones((23,64),np.float32)/8
    k1=rng.uniform(0,100,(17,2));k2=rng.uniform(0,100,(23,2))
    r=relation_match(x,y,k1,k2,(100,100),(100,100),cfg,trace=True)
    assert r['fallback'] and r['compared_pairs']==17*23
    for a,b in zip(r['matches'],full_cosine(x,y)):np.testing.assert_array_equal(a,b)
    for n,m in [(0,0),(0,5),(5,0),(1,1)]:
        x=np.zeros((n,64),np.float32);y=np.zeros((m,64),np.float32)
        r=relation_match(x,y,np.zeros((n,2)),np.zeros((m,2)),(100,100),(100,100),cfg,True)
        for a,b in zip(r['matches'],full_cosine(x,y)):np.testing.assert_array_equal(a,b)
    print('PASS: shift/rotation/shrink/combined, known correspondences, masked cosine oracle, fallback/ties/empty, no duplicate pairs.')
check_relation_implementation()


PASS: shift/rotation/shrink/combined, known correspondences, masked cosine oracle, fallback/ties/empty, no duplicate pairs.


## 6. Calibration에서 confidence threshold 고정
정답을 사용하는 곳은 이 calibration과 마지막 정확도 평가뿐입니다. 양방향 검사에 쓰는 geometry threshold와 probe 수는 HPatches에서 튜닝하지 않습니다.

In [6]:
import pandas as pd
from threadpoolctl import threadpool_limits,threadpool_info
import platform,os,json,hashlib,sys
from datetime import datetime
from zoneinfo import ZoneInfo

def array32(x):
    return np.ascontiguousarray(x.detach().cpu().numpy() if hasattr(x,'detach') else x,dtype=np.float32)
def pack_pair(p):
    return dict(x=array32(p['d1']),y=array32(p['d2']),k1=array32(p['k1']),k2=array32(p['k2']),
                shape1=p['shape1'],shape2=p['shape2'],H=np.asarray(p['H']),name=p.get('name',''))
def reprojection_error(p,ij):
    i,j=ij;h=np.c_[p['k1'][i],np.ones(len(i))]@p['H'].T
    with np.errstate(invalid='ignore',divide='ignore'):
        err=np.linalg.norm(h[:,:2]/h[:,2:]-p['k2'][j],axis=1)
    return np.where(np.isfinite(err),err,np.inf)

# Use ONLY calibration scenes to set descriptor confidence; no test-set tuning.
def calibrate_confidence(pairs):
    scores=[];margins=[]
    with threadpool_limits(limits=CPU_THREADS):
        for raw in pairs:
            p=pack_pair(raw)
            if len(p['y'])<2 or not len(p['x']):continue
            s=p['x']@p['y'].T;j=s.argmax(1);i=np.arange(len(j))
            e=reprojection_error(p,(i,j));correct=e<GEOMETRIC_THRESHOLD
            best=s[i,j];second=np.partition(s,-2,axis=1)[:,-2]
            scores.extend(best[correct].tolist());margins.extend((best-second)[correct].tolist())
    if len(scores)<32:return RelationConfig(),dict(source='fixed_defaults_insufficient_calibration',correct_samples=len(scores))
    cfg=RelationConfig(cosine_min=float(np.clip(np.quantile(scores,.10),.2,.8)),
                       margin_min=float(np.clip(np.quantile(margins,.10),.005,.08)))
    return cfg,dict(source='calibration_correct_top1_10th_percentile_clipped',correct_samples=len(scores))

CONFIG,calibration_info=calibrate_confidence(calibration_pairs)
print('Frozen configuration:',asdict(CONFIG))
print('Descriptor threshold calibration:',calibration_info)
print('Geometry/probe thresholds are fixed experimental defaults, NOT tuned on HPatches.')


Frozen configuration: {'anchor_quadrant': 0, 'cosine_min': 0.8, 'margin_min': 0.016104817390441895, 'min_seed': 8, 'geom_tol': 0.035, 'region_padding': 0.04, 'min_seed_fraction': 0.45, 'min_seed_span': 0.08, 'probes': 16, 'min_probe_matches': 4, 'min_probe_fraction': 0.6, 'ransac_trials': 96, 'min_scale': 0.5, 'max_scale': 2.0, 'seed': 7}
Descriptor threshold calibration: {'source': 'calibration_correct_top1_10th_percentile_clipped', 'correct_samples': 5273}
Geometry/probe thresholds are fixed experimental defaults, NOT tuned on HPatches.


## 7. 같은 입력에서 원본과 비교·시간 분해·결과 저장
5쌍/데이터셋, warmup 5, 3라운드×12회, 방법 순서 섞기. 표의 p10/p90은 pair별 percentile의 중앙값입니다. 정확도는 모든 평가 쌍에서 측정합니다. 반복 측정 원본과 입력 NPZ도 새 결과 폴더에 저장합니다. 이 셀만 다시 실행해도 됩니다.

In [7]:
DATASETS={'synthetic_validation':[pack_pair(p) for p in validation_pairs]}
if real_pairs:DATASETS['HPatches_test']=[pack_pair(p) for p in real_pairs]
accuracy=[];details=[];raw_times=[];stage_times=[];all_errors={};checks=[]

def run_relation(p,trace=False):
    return relation_match(p['x'],p['y'],p['k1'],p['k2'],p['shape1'],p['shape2'],CONFIG,trace)

def baseline_stages(p):
    t=time.perf_counter_ns();s=p['x']@p['y'].T;t1=time.perf_counter_ns()
    ij=mnn_scores(s);t2=time.perf_counter_ns()
    return ij,dict(score_matrix_ms=(t1-t)/1e6,final_mnn_ms=(t2-t1)/1e6)

with threadpool_limits(limits=CPU_THREADS):
 for dataset,pairs in DATASETS.items():
  sample=set(np.unique(np.linspace(0,len(pairs)-1,min(TIMING_PAIRS_PER_DATASET,len(pairs)),dtype=int))) if pairs else set()
  for pi,p in enumerate(pairs):
    ref=full_cosine(p['x'],p['y']);ref_err=reprojection_error(p,ref)
    out=run_relation(p,trace=True);got=out['matches'];mask=out['edges']
    # The oracle is outside matching/timing. It cannot influence the candidate graph.
    dense=p['x']@p['y'].T
    np.testing.assert_allclose(out['scores'][mask],dense[mask],rtol=1e-5,atol=2e-6)
    masked=np.where(mask,dense,-np.inf)
    oracle=mnn_scores(masked)
    # BLAS shapes can differ by tiny roundoff; log exact agreement instead of hiding it.
    same=all(np.array_equal(a,b) for a,b in zip(got,oracle))
    np.testing.assert_array_equal(mnn_scores(out['scores'])[0],got[0])
    assert int(mask.sum())==out['compared_pairs']<=len(p['x'])*len(p['y'])
    check=dict(dataset=dataset,pair=pi,no_duplicate_pairs=True,masked_full_blas_mnn_equal=same,
               pair_values_close=True,fallback_full_coverage=bool(mask.all()) if out['fallback'] else None)
    if out['fallback']:assert mask.all()
    checks.append(check)
    ref_set=set(zip(*ref));ref_correct=set(zip(ref[0][ref_err<GEOMETRIC_THRESHOLD],ref[1][ref_err<GEOMETRIC_THRESHOLD]))
    for method,ij in [('float_cosine',ref),('quadrant_relation',got)]:
        e=reprojection_error(p,ij);matched=set(zip(*ij));n,m=len(p['x']),len(p['y'])
        row=dict(dataset=dataset,pair=pi,name=p['name'],method=method,n=n,m=m,mutual=len(e),
            correct=int((e<GEOMETRIC_THRESHOLD).sum()),reference_mutual=len(ref_set),
            reference_correct=len(ref_correct),retained_reference_mnn=len(matched&ref_set),
            retained_reference_correct=len(matched&ref_correct),
            reference_correct_candidates=sum(mask[i,j] for i,j in ref_correct) if method=='quadrant_relation' else len(ref_correct),
            compared_pairs=out['compared_pairs'] if method=='quadrant_relation' else n*m,total_pairs=n*m,
            fallback=out['fallback'] if method=='quadrant_relation' else False,
            estimated_dot_flops=128*(out['compared_pairs'] if method=='quadrant_relation' else n*m),
            score_cache_bytes=out['cache_bytes'] if method=='quadrant_relation' else 4*n*m,
            max_score_block_bytes=out['max_score_block_bytes'] if method=='quadrant_relation' else 4*n*m,
            descriptor_bytes=p['x'].nbytes+p['y'].nbytes,
            matmul_calls=out['matmul_calls'] if method=='quadrant_relation' else 1)
        accuracy.append(row);all_errors.setdefault((dataset,method),[]).extend(e.tolist())
    details.append(dict(dataset=dataset,pair=pi,name=p['name'],anchor_quadrant=out['anchor']+1,
        anchor_kps=out['anchor_kps'],probe_kps=out['probe_kps'],fallback_reason=out['fallback_reason'],
        models=out['models'],allowed_quadrants=out['graph'].astype(int).tolist()))
    if pi in sample and len(p['x']) and len(p['y']):
        # No trace mask/oracle/geometric accuracy inside timing.
        for _ in range(WARMUP):baseline_stages(p);run_relation(p)
        rng_t=np.random.default_rng(SEED+pi)
        for round_id in range(TIMING_ROUNDS):
          for repeat in range(REPEATS):
            for method in rng_t.permutation(['float_cosine','quadrant_relation']):
                start=time.perf_counter_ns()
                if method=='float_cosine':_,stages=baseline_stages(p)
                else:measured=run_relation(p);stages=measured['stages']
                ms=(time.perf_counter_ns()-start)/1e6
                raw_times.append(dict(dataset=dataset,pair=pi,method=str(method),round=round_id,repeat=repeat,total_ms=ms))
                for stage,value in stages.items():stage_times.append(dict(dataset=dataset,pair=pi,method=str(method),round=round_id,repeat=repeat,stage=stage,ms=value))
    print('Evaluated:',dataset,pi+1,'/',len(pairs),'pairs',out['compared_pairs'],'/',len(p['x'])*len(p['y']),
          'fallback:',out['fallback_reason'] or 'no')
 cpu_threadpools=threadpool_info()

accuracy_df=pd.DataFrame(accuracy);timing_raw=pd.DataFrame(raw_times);stages_raw=pd.DataFrame(stage_times)
summary=[]
for (dataset,method),g in accuracy_df.groupby(['dataset','method'],sort=False):
    ratio=lambda a,b:100*float(a)/float(b) if b else np.nan
    err=all_errors[dataset,method]
    summary.append(dict(dataset=dataset,method=method,mutual=int(g.mutual.sum()),correct=int(g.correct.sum()),
        precision_pct=ratio(g.correct.sum(),g.mutual.sum()),median_px=float(np.median(err)) if len(err) else np.nan,
        comparison_pct=ratio(g.compared_pairs.sum(),g.total_pairs.sum()),
        reference_mnn_keep_pct=ratio(g.retained_reference_mnn.sum(),g.reference_mutual.sum()),
        reference_correct_keep_pct=ratio(g.retained_reference_correct.sum(),g.reference_correct.sum()),
        correct_candidate_keep_pct=ratio(g.reference_correct_candidates.sum(),g.reference_correct.sum()),
        fallback_pct=100*g.fallback.mean(),median_matmul_calls=g.matmul_calls.median(),
        median_score_cache_KiB=g.score_cache_bytes.median()/1024))
summary_df=pd.DataFrame(summary)
if len(timing_raw):
    pair_times=timing_raw.groupby(['dataset','pair','method']).total_ms.agg(
        median_ms='median',p10_ms=lambda s:s.quantile(.1),p90_ms=lambda s:s.quantile(.9)).reset_index()
    ts=pair_times.groupby(['dataset','method'])[['median_ms','p10_ms','p90_ms']].median().reset_index()
    summary_df=summary_df.merge(ts,on=['dataset','method'])
    for dataset,g in summary_df.groupby('dataset'):
        baseline=float(g[g.method=='float_cosine'].median_ms.iloc[0]);mask=summary_df.dataset==dataset
        summary_df.loc[mask,'speedup_vs_float']=baseline/summary_df.loc[mask,'median_ms']
    stages_summary=stages_raw.groupby(['dataset','pair','method','stage']).ms.median().groupby(['dataset','method','stage']).median().reset_index()
else:pair_times=pd.DataFrame();stages_summary=pd.DataFrame()
display(summary_df.round(4));display(stages_summary.round(4))
print('Precision is geometric precision@3 resized pixels; reference keep is retention of baseline matches, NOT ground-truth recall.')
print('Total includes cache/layout, seed search, fitting, reverse validation, remaining search and MNN; XFeat/calibration/checks/I/O excluded.')
print('Comparison count includes seed + probes + remaining/fallback, each pair at most once. Dense score cache retained: no memory compression claim.')
print('p10/p90 are medians of per-pair timing percentiles, NOT confidence intervals. Inspect raw round-to-round drift.')
print('Masked full-BLAS exact MNN checks:',sum(c['masked_full_blas_mnn_equal'] for c in checks),'/',len(checks))

run_id=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
result_dir=data_dir/'experiment_results'/('quadrant_relation_'+run_id);result_dir.mkdir(parents=True,exist_ok=False)
for name,df in [('summary.csv',summary_df),('accuracy_per_pair.csv',accuracy_df),('timing_raw.csv',timing_raw),
                ('timing_per_pair.csv',pair_times),('stages_raw.csv',stages_raw),('stages_summary.csv',stages_summary),
                ('checks.csv',pd.DataFrame(checks))]:df.to_csv(result_dir/name,index=False)
cpu_model=platform.processor()
try:
    cpu_model=next(line.split(':',1)[1].strip() for line in open('/proc/cpuinfo') if line.startswith('model name'))
except (OSError,StopIteration):pass
try:ram_gib=os.sysconf('SC_PHYS_PAGES')*os.sysconf('SC_PAGE_SIZE')/2**30
except (ValueError,OSError,AttributeError):ram_gib=None
metadata=dict(config=asdict(CONFIG),calibration=calibration_info,seed=SEED,top_k=TOP_K,
    cpu=cpu_model,platform=platform.platform(),logical_cpus=os.cpu_count(),host_ram_GiB=ram_gib,cpu_threads=CPU_THREADS,
    numpy=np.__version__,python=sys.version,torch=torch.__version__,opencv=cv2.__version__,
    threadpools=cpu_threadpools,warmup=WARMUP,repeats=REPEATS,timing_rounds=TIMING_ROUNDS,
    timing_pairs_per_dataset=TIMING_PAIRS_PER_DATASET,geometry_threshold_px=GEOMETRIC_THRESHOLD,
    summary=json.loads(summary_df.to_json(orient='records')),details=details,
    notes=['Only float baseline and spatial relation matching. No Thermometer/Grouping/Blob training.',
           'Homography never passed to matcher; only calibration and accuracy evaluation.',
           'Similarity model allows joint rotation/scale/translation; not a general perspective model.',
           'Both accepted hypotheses use union; failed validation triggers full-search fallback.',
           'Score cache and max block metrics are array sizes, not process peak memory.'])
(result_dir/'results.json').write_text(json.dumps(metadata,indent=2,allow_nan=False))
# Save exact evaluation inputs for a repeatable timing-only run later (no extraction needed).
for dataset,pairs in DATASETS.items():
    payload={}
    for pi,p in enumerate(pairs):
        for key in ['x','y','k1','k2','H','shape1','shape2']:payload[f'{pi}_{key}']=np.asarray(p[key])
    np.savez_compressed(result_dir/(dataset+'_inputs.npz'),**payload)
print('CPU:',cpu_model,'| logical CPUs:',os.cpu_count(),'| thread limit:',CPU_THREADS,'| host RAM GiB:',ram_gib)
print('NumPy:',np.__version__,'| Torch:',torch.__version__,'| OpenCV:',cv2.__version__)
print('RESULT FOLDER:',result_dir)


Evaluated: synthetic_validation 1 / 18 pairs 640000 / 640000 fallback: no
Evaluated: synthetic_validation 2 / 18 pairs 640000 / 640000 fallback: no
Evaluated: synthetic_validation 3 / 18 pairs 640000 / 640000 fallback: no
Evaluated: synthetic_validation 4 / 18 pairs 640000 / 640000 fallback: no
Evaluated: synthetic_validation 5 / 18 pairs 601104 / 640000 fallback: no
Evaluated: synthetic_validation 6 / 18 pairs 640000 / 640000 fallback: reverse_validation_failed
Evaluated: synthetic_validation 7 / 18 pairs 640000 / 640000 fallback: no
Evaluated: synthetic_validation 8 / 18 pairs 640000 / 640000 fallback: no
Evaluated: synthetic_validation 9 / 18 pairs 640000 / 640000 fallback: no
Evaluated: synthetic_validation 10 / 18 pairs 640000 / 640000 fallback: reverse_validation_failed
Evaluated: synthetic_validation 11 / 18 pairs 640000 / 640000 fallback: reverse_validation_failed
Evaluated: synthetic_validation 12 / 18 pairs 640000 / 640000 fallback: no
Evaluated: synthetic_validation 13 / 18 

,dataset,method,mutual,correct,precision_pct,median_px,comparison_pct,reference_mnn_keep_pct,reference_correct_keep_pct,correct_candidate_keep_pct,fallback_pct,median_matmul_calls,median_score_cache_KiB,median_ms,p10_ms,p90_ms,speedup_vs_float
0,synthetic_validation,float_cosine,8129,5613,69.0491,1.3484,100.0000,100.0000,100.0,100.0,0.0000,1.0,2500.0,3.6283,3.4574,4.1279,1.0000
1,synthetic_validation,quadrant_relation,8128,5613,69.0576,1.3481,98.1476,99.9877,100.0,100.0,16.6667,5.0,2500.0,29.6055,29.2463,34.1471,0.1226
2,HPatches_test,float_cosine,10607,7138,67.2952,1.1764,100.0000,100.0000,100.0,100.0,0.0000,1.0,2500.0,3.5523,3.4432,4.1040,1.0000
3,HPatches_test,quadrant_relation,10607,7138,67.2952,1.1764,100.0000,100.0000,100.0,100.0,50.0000,5.0,2500.0,29.8921,28.3712,36.1050,0.1188


,dataset,method,stage,ms
0,HPatches_test,float_cosine,final_mnn_ms,2.0280
1,HPatches_test,float_cosine,score_matrix_ms,1.5039
2,HPatches_test,quadrant_relation,anchor_search_ms,2.1558
3,HPatches_test,quadrant_relation,final_mnn_ms,2.2290
4,HPatches_test,quadrant_relation,hypothesis_fit_ms,17.4121
5,HPatches_test,quadrant_relation,layout_cache_ms,0.4191
6,HPatches_test,quadrant_relation,remaining_search_ms,7.3356
7,HPatches_test,quadrant_relation,reverse_probe_search_ms,0.4408
8,HPatches_test,quadrant_relation,validation_graph_ms,0.4684
9,synthetic_validation,float_cosine,final_mnn_ms,2.0631


Precision is geometric precision@3 resized pixels; reference keep is retention of baseline matches, NOT ground-truth recall.
Total includes cache/layout, seed search, fitting, reverse validation, remaining search and MNN; XFeat/calibration/checks/I/O excluded.
Comparison count includes seed + probes + remaining/fallback, each pair at most once. Dense score cache retained: no memory compression claim.
p10/p90 are medians of per-pair timing percentiles, NOT confidence intervals. Inspect raw round-to-round drift.
Masked full-BLAS exact MNN checks: 42 / 42
CPU: Intel(R) Xeon(R) CPU @ 2.20GHz | logical CPUs: 2 | thread limit: 2 | host RAM GiB: 12.671417236328125
NumPy: 2.1.3 | Torch: 2.11.0+cpu | OpenCV: 5.0.0
RESULT FOLDER: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/experiment_results/quadrant_relation_20261006_152729_168298
